###  Create the .env Configuration

In [2]:
with open(".env", "w") as f:
    f.write("PASS_MARK=40\n")
    f.write("FINE_PER_DAY=10\n")
    f.write("UNIVERSITY_NAME=Global Tech University")
print(".env file created successfully!")

.env file created successfully!


### Sample CSV Data

In [3]:
import csv
data = [
    ["id", "name", "marks", "due_date"],
    ["S001", "Alice Johnson", "85", "2023-10-01"],
    ["S002", "Bob Smith", "35", "2023-10-05"],     # Failed student
    ["S003", "Charlie Brown", "invalid", "2023-10-10"], # Bad data
    ["S004", "David Miller", "72", "2023-09-20"],    # Late student
    ["S005", "Eve Adams", "-10", "2023-10-15"]       # Validation error
]

with open('students.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(data)

print("students.csv created successfully!")

students.csv created successfully!


### Custom Exceptions and Decorators

In [4]:
import functools
import time
class StudentPortalError(Exception):
    """Base class for exceptions in this module."""
    pass

class ValidationError(StudentPortalError):
    """Raised when data validation fails."""
    pass
def log_action(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] Executing: {func.__name__}...")
        result = func(*args, **kwargs)
        return result
    return wrapper

### The Student Class

In [5]:
from datetime import datetime, date
class Student:
    def __init__(self, sid, name, marks, due_date, pass_mark, fine_rate):
        self.sid = sid
        self.name = name
        self.marks = self.validate_marks(marks)
        self.due_date = datetime.strptime(due_date, "%Y-%m-%d").date()
        self.pass_mark = int(pass_mark)
        self.fine_rate = int(fine_rate)
    def validate_marks(self, marks):
        try:
            m = float(marks)
            if not (0 <= m <= 100):
                raise ValidationError(f"Invalid range for {self.sid}: {m}")
            return m
        except ValueError:
            raise ValidationError(f"Invalid numeric value for {self.sid}: {marks}")
    @property
    def status(self):
        return "PASS" if self.marks >= self.pass_mark else "FAIL"

    def calculate_fine(self):
        today = date.today()
        if today > self.due_date:
            days_late = (today - self.due_date).days
            return days_late * self.fine_rate
        return 0

    def __str__(self):
        return f"ID: {self.sid} | Name: {self.name} | Status: {self.status} | Fine: ${self.calculate_fine()}"

### The Portal System

In [6]:
import asyncio
import os
from dotenv import load_dotenv

load_dotenv() 

class AcademicPortal:
    def __init__(self):
        self.students = []
        self.pass_mark = os.getenv("PASS_MARK")
        self.fine_rate = os.getenv("FINE_PER_DAY")

    @log_action
    def load_data(self, file_path):
        with open(file_path, mode='r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                try:
                    student = Student(
                        row['id'], row['name'], row['marks'], 
                        row['due_date'], self.pass_mark, self.fine_rate
                    )
                    self.students.append(student)
                except (ValidationError, ValueError) as e:
                    print(f"[ERROR] Skipping record: {e}")

    async def generate_report_task(self, student):
        """Simulates an async task like generating a PDF or sending email."""
        await asyncio.sleep(0.5)
        return f"Report generated for {student.name} - Status: {student.status}"

    @log_action
    async def process_all_reports(self):
        print(f"\n--- Generating Timestamped Reports ({datetime.now()}) ---")
        tasks = [self.generate_report_task(s) for s in self.students]
        results = await asyncio.gather(*tasks)
        for res in results:
            print(res)

    def display_summary(self):
        print("\n--- Student Summary ---")
        for s in self.students:
            print(s)

### Main Execution

In [7]:
# Cell 6: Run the Application
async def main():
    portal = AcademicPortal()
    
    # 1. Load and Validate Data
    portal.load_data('students.csv')
    
    # 2. Display Student Summary (Fines & Pass/Fail)
    portal.display_summary()
    
    # 3. Process Reports Concurrently
    await portal.process_all_reports()

# Entry point for Jupyter
if __name__ == "__main__":
    # In Jupyter, we use await main() directly or use this helper:
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.run(main())

[LOG] Executing: load_data...
[ERROR] Skipping record: Invalid numeric value for S003: invalid
[ERROR] Skipping record: Invalid range for S005: -10.0

--- Student Summary ---
ID: S001 | Name: Alice Johnson | Status: PASS | Fine: $8900
ID: S002 | Name: Bob Smith | Status: FAIL | Fine: $8860
ID: S004 | Name: David Miller | Status: PASS | Fine: $9010
[LOG] Executing: process_all_reports...

--- Generating Timestamped Reports (2026-03-09 00:04:52.023282) ---
Report generated for Alice Johnson - Status: PASS
Report generated for Bob Smith - Status: FAIL
Report generated for David Miller - Status: PASS
